# Man-in-the-Middle — Hijack

Here the **attacker itself** performs the intervention, not the GCS. The MITM
watches the telemetry it is relaying and, once the drone reaches a chosen
waypoint (`MISSION_CURRENT.seq >= trigger_seq`), it injects
`SET_MODE(GUIDED)` + `DO_REPOSITION` toward attacker-chosen coordinates. The
injected commands are spoofed to look like they came from the GCS (sysid 255),
so the onboard Logic forwards them to the flight controller as legitimate.

This is a *visible* hijack: telemetry still flows to the GCS, so the operator
watches the drone get redirected. The GCS sends no commands of its own here —
`simulator.intervention` is left unset.

In [1]:
from simulator import Simulator
from simulator.config import PARAMS_PATH, Color, Model
from simulator.entities import SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin and waypoints

In [2]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, 0, 0, 0)
speed = 5.0    # m/s
cruise_alt = 10.0  # m
model = Model.IRIS
sysid = 1

# Mission seq: seq=0 home, seq=1 TAKEOFF, seq=2 flying to north_100,
#              seq=3 flying to north_200 ← attacker hijacks here
home_wp    = ENU(x=0, y=0,   z=0)
north_100  = ENU(x=0, y=100, z=cruise_alt)
north_200  = ENU(x=0, y=200, z=cruise_alt)
mission_wps = [home_wp, north_100, north_200]

# Attacker's chosen destination: 100 m WEST of origin (opposite the mission).
hijack_target = gra_origin.unpose().to_abs(ENU(x=-100, y=0, z=cruise_alt))
print(f"Hijack target: lat={hijack_target.lat:.7f}, lon={hijack_target.lon:.7f}")

Hijack target: lat=-35.3633280, lon=149.1641238


## Vehicle

In [3]:
mission_path = "simulator/planner/missions/mitm_north.waypoints"

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=mission_path,
    navigation_speed=speed,
    firmware=model.firmware,
)

vehicle = SimVehicle.from_relative(
    sysid=sysid,
    gcs_name=f"BLUE_{Color.BLUE.emoji}",
    plan=plan,
    color=Color.BLUE,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    model=model,
)

## Visualizer

In [4]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
gaz.markers.append(origin_marker)

## Simulator + MITM hijack

In [ ]:
simulator = Simulator(visualizer=gaz, verbose=1)
simulator.add_vehicle(vehicle, parm=str(PARAMS_PATH / "vehicle.parm"))

# No GCS intervention here — the attacker does the redirection.
# `simulator.mitm` is keyed by sysid, so this targets only this vehicle —
# other vehicles sharing this GCS (if any) would be unaffected.
simulator.mitm[sysid] = {
    "strategy": "hijack",
    "params": {
        "trigger_seq": 3,
        "target_lat": hijack_target.lat,
        "target_lon": hijack_target.lon,
        "target_alt": cruise_alt,
    },
}

simulator.show()

In [ ]:
orac = simulator.launch()
orac.run()

14:19:29 - Oracle ⚪ - INFO - 🖥️  Gazebo launched for realistic simulation and 3D visualization.
14:19:29 - Oracle ⚪ - INFO - 🚀 GCS BLUE_🟦 launched (PID 4879)
14:19:29 - Oracle ⚪ - INFO - 🏁 Starting Oracle with 1 vehicles and 1 GCSs
14:19:31 - mitm_1 - INFO - MITM proxy active for vehicle 1 (strategy=HijackStrategy)
14:19:32 - GCS_BLUE_🟦 - INFO - Vehicle 1 connected
14:19:32 - GCS_BLUE_🟦 - INFO -  GCS BLUE_🟦 started with 1 Vehicles
14:19:32 - logic_1 - INFO - Logic 🧠 1: launching
14:19:32 - GCS_BLUE_🟦 - INFO - Monitoring Vehicle 1
14:19:32 - logic_1 - INFO - 🧹 Vehicle 1: Cleared previous mission
14:19:33 - logic_1 - INFO - ✅ Vehicle 1: Mission successfully loaded!
14:19:33 - logic_1 - INFO - ✅ Vehicle 1: Action Done: 💾 UPLOAD_MISSION
14:20:13 - logic_1 - INFO - ✅ Vehicle 1: Action Done: 🛡️ PREARM
14:20:13 - logic_1 - INFO - ✅ Vehicle 1: Action Done: ⚙️ MODE
14:20:13 - logic_1 - INFO - ✅ Vehicle 1: Action Done: 🔒 ARM
14:20:13 - logic_1 - INFO - 🚀 Vehicle 1: Mission has started
14:20:13 -

## What to observe

- The drone flies north, then **turns west** mid-mission — driven entirely by
  the man-in-the-middle, with no command from the GCS.
- `simulator/logs/mitm/mitm_1.log` — `MITM hijack: redirecting vehicle 1 ...`.
- `simulator/logs/logics/logic_1.log` — `GCS→SITL forwarding SET_MODE` /
  `COMMAND_INT` (Logic forwards the spoofed commands to the flight controller).
- `simulator/logs/GCSs/GCS_BLUE_*.log` — **no** `GCS intervention` line; the
  redirect did not originate from the GCS.